# Geneformer parameter- and quality-scaling on PBMC (N=100k)

Loads the CSV produced by `2026-04-16_14-42_compute_scaling_params_geneformer_pbmc.py`
(5 architectures log-spaced from ~1.5M to ~100M params × 10 qualities) and plots
test MLM loss vs measurement quality, with one line per model size.

Expected CSV columns: `name, quality, hidden_size, num_hidden_layers, num_attention_heads, intermediate_size, total_params, non_embedding_params, final_train_loss, final_eval_loss, best_eval_loss, test_loss, train_time_s, ...`

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

CSV_PATH = Path("2026-04-16_14-42_scaling_params_geneformer_pbmc_results.csv")
assert CSV_PATH.exists(), f"Results CSV not found at {CSV_PATH.resolve()} — run the .py script first."
df = pd.read_csv(CSV_PATH)
df = df.sort_values(["total_params", "quality"]).reset_index(drop=True)
print(f"{len(df)} rows | {df['name'].nunique()} configs × {df['quality'].nunique()} qualities")
df.head(12)

## Test loss vs parameter count, one line per quality

Each line is a fixed measurement quality; the x-axis walks through architecture sizes. This is the "parameter scaling curve, sliced by noise".

In [ ]:
def plot_params_curves(ax, x_col: str, title: str):
    qualities = sorted(df["quality"].unique())
    cmap = plt.get_cmap("viridis")
    for i, q in enumerate(qualities):
        sub = df[df["quality"] == q].sort_values(x_col)
        color = cmap(i / max(1, len(qualities) - 1))
        ax.plot(sub[x_col], sub["test_loss"], marker="o", color=color, label=f"q={q:.4g}")
    ax.set_xscale("log")
    ax.set_yscale("log")
    ax.set_xlabel(x_col)
    ax.set_ylabel("test MLM loss")
    ax.set_title(title)
    ax.grid(True, which="both", alpha=0.3)
    ax.legend(fontsize=7, ncol=2)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
plot_params_curves(axes[0], "total_params", "Test loss vs total params — coloured by quality")
plot_params_curves(axes[1], "non_embedding_params", "Test loss vs non-embedding params — coloured by quality")
plt.tight_layout()
plt.show()

## Test loss vs quality, one line per architecture

Transpose of the previous view — each line is a fixed architecture, the x-axis is the downsampling quality.

In [ ]:
fig, ax = plt.subplots(figsize=(8, 5))
configs_by_size = df.drop_duplicates("name").sort_values("total_params")[["name", "total_params"]].values
cmap = plt.get_cmap("plasma")
for i, (name, n_params) in enumerate(configs_by_size):
    sub = df[df["name"] == name].sort_values("quality")
    color = cmap(i / max(1, len(configs_by_size) - 1))
    ax.plot(sub["quality"], sub["test_loss"], marker="o", color=color,
            label=f"{name} ({n_params/1e6:.1f}M)")
ax.set_xscale("log")
ax.set_yscale("log")
ax.set_xlabel("quality (downsampling fraction)")
ax.set_ylabel("test MLM loss")
ax.set_title("Test loss vs quality — coloured by model size")
ax.grid(True, which="both", alpha=0.3)
ax.legend(fontsize=8, ncol=2)
plt.tight_layout()
plt.show()

## Heatmap: test loss over (params × quality)

Full 2D view of the sweep.

In [ ]:
pivot = df.pivot_table(index="name", columns="quality", values="test_loss", aggfunc="mean")
name_order = df.drop_duplicates("name").sort_values("total_params")["name"].tolist()
pivot = pivot.reindex(name_order)
pivot = pivot[sorted(pivot.columns)]

fig, ax = plt.subplots(figsize=(10, 5))
im = ax.imshow(pivot.values, aspect="auto", cmap="viridis")
ax.set_xticks(range(len(pivot.columns)))
ax.set_xticklabels([f"{q:.4g}" for q in pivot.columns], rotation=45, ha="right")
ax.set_yticks(range(len(pivot.index)))
ax.set_yticklabels([f"{n}\n{df[df['name']==n]['total_params'].iloc[0]/1e6:.1f}M" for n in pivot.index])
ax.set_xlabel("quality")
ax.set_ylabel("architecture")
ax.set_title("test MLM loss (lower = better)")
for i in range(pivot.shape[0]):
    for j in range(pivot.shape[1]):
        v = pivot.values[i, j]
        if np.isfinite(v):
            ax.text(j, i, f"{v:.2f}", ha="center", va="center",
                    color="white" if v > np.nanmean(pivot.values) else "black", fontsize=7)
plt.colorbar(im, ax=ax, label="test loss")
plt.tight_layout()
plt.show()

## Power-law fits: one α per quality

Fit `test_loss ≈ A · N^α` (log-log regression) separately at each quality level and see how the scaling exponent α depends on measurement noise.

In [ ]:
def fit_power_law(x, y):
    x = np.asarray(x); y = np.asarray(y)
    mask = np.isfinite(x) & np.isfinite(y) & (x > 0) & (y > 0)
    if mask.sum() < 2:
        return np.nan, np.nan
    slope, intercept = np.polyfit(np.log(x[mask]), np.log(y[mask]), 1)
    return slope, np.exp(intercept)

fit_rows = []
for q, sub in df.groupby("quality"):
    sub = sub.sort_values("total_params")
    s_tot, A_tot = fit_power_law(sub["total_params"].values, sub["test_loss"].values)
    s_nemb, A_nemb = fit_power_law(sub["non_embedding_params"].values, sub["test_loss"].values)
    fit_rows.append({"quality": q, "alpha_total": -s_tot, "A_total": A_tot,
                     "alpha_nonemb": -s_nemb, "A_nonemb": A_nemb})
fits = pd.DataFrame(fit_rows).sort_values("quality").reset_index(drop=True)
fits

In [ ]:
fig, ax = plt.subplots(figsize=(7, 4))
ax.plot(fits["quality"], fits["alpha_total"], "o-", label="α (total params)")
ax.plot(fits["quality"], fits["alpha_nonemb"], "s-", label="α (non-embedding)")
ax.set_xscale("log")
ax.set_xlabel("quality")
ax.set_ylabel("scaling exponent α   (test_loss ∝ N^-α)")
ax.set_title("How the parameter-scaling exponent depends on quality")
ax.grid(True, which="both", alpha=0.3)
ax.legend()
plt.tight_layout()
plt.show()

## Train time vs parameters

Sanity check — larger models should train slower. Coloured by quality in case tokenized sequence length varies with downsampling.

In [ ]:
if "train_time_s" in df.columns:
    fig, ax = plt.subplots(figsize=(7, 5))
    qualities = sorted(df["quality"].unique())
    cmap = plt.get_cmap("viridis")
    for i, q in enumerate(qualities):
        sub = df[df["quality"] == q].sort_values("total_params")
        ax.plot(sub["total_params"], sub["train_time_s"], marker="o",
                color=cmap(i / max(1, len(qualities) - 1)), label=f"q={q:.4g}")
    ax.set_xscale("log")
    ax.set_yscale("log")
    ax.set_xlabel("total parameters")
    ax.set_ylabel("wall-clock train time (s)")
    ax.set_title("Train time vs model size — coloured by quality")
    ax.grid(True, which="both", alpha=0.3)
    ax.legend(fontsize=7, ncol=2)
    plt.tight_layout()
    plt.show()
else:
    print("train_time_s not present in the CSV yet.")

## Summary table

In [ ]:
cols = ["name", "quality", "hidden_size", "num_hidden_layers", "num_attention_heads",
        "intermediate_size", "total_params", "non_embedding_params",
        "final_train_loss", "best_eval_loss", "test_loss"]
df[[c for c in cols if c in df.columns]]